# Fed-batch yeast: which feeding policy pins down the growth kinetics? D-, A-, E- and modified-E-optimal designs

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zuhairblr/middoe/blob/main/tests/tutorials/04_design_precision/a_criteria/tutorial.ipynb)

**Model-based design of experiments for parameter precision** (`mbdoe_pp`).

After two preliminary runs the four growth parameters are known roughly. `mbdoe_pp` chooses the
next run (the dilution-rate and feed profiles, the initial biomass and the sampling times) that
maximises a measure of the Fisher information matrix:

| criterion | maximises / minimises | in words |
|---|---|---|
| **D** | det(FIM) | the smallest joint confidence region |
| **A** | trace(FIM⁻¹) | the smallest average variance |
| **E** | λ_min(FIM) | the shortest longest axis |
| **ME** | λ_max / λ_min | the roundest region (balanced precision) |

You will design one run with each criterion, compare them, run the D-optimal one in silico and
see the intervals shrink.

In [ ]:
# Google Colab only: install MIDDoE and fetch the model file(s); does nothing elsewhere
import sys
if 'google.colab' in sys.modules:
    import subprocess, urllib.request
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'git+https://github.com/zuhairblr/middoe.git'], check=True)
    for f in ['model.py']:
        urllib.request.urlretrieve('https://raw.githubusercontent.com/zuhairblr/middoe/main/tests/tutorials/04_design_precision/a_criteria/' + f, f)

### The process

A fed-batch baker's-yeast fermentation (Monod growth with maintenance):

$$r = \frac{\mu_{max}\, y_2}{K_s + y_2},\qquad \frac{dy_1}{dt} = (r - u_1 - m)\,y_1,\qquad
\frac{dy_2}{dt} = -\frac{r\,y_1}{Y_{xs}} + u_1 (u_2 - y_2)$$

Biomass `y1` and substrate `y2` are measured; the dilution rate `u1(t)` and the feed concentration
`u2(t)` can change during the run, the initial biomass `y10` is chosen per run. Four parameters:
`mu_max`, `Ks`, `Yxs`, `m`.

In [ ]:
from pathlib import Path
import shutil
import numpy as np
import pandas as pd

from middoe.krnl_expera import expera
from middoe.iden_parmest import parmest
from middoe.iden_uncert import uncert
from middoe.log_utils import save_rounds
from middoe.des_pp import mbdoe_pp

QUICK = False   # True: tiny budgets and fewer designs, a check that everything runs (minutes), not a result.
HERE = Path.cwd()
# Start clean: expera() ADDS sheets to an existing data.xlsx, so remove what an earlier run wrote.
for old in ['data.xlsx', 'data_plots', 'modelling', 'confidence', 'post_processing', 'estimability',
            'meshindep_plots', 'design_plots', 'iden_results.jac']:
    p = HERE / old
    shutil.rmtree(p) if p.is_dir() else p.unlink(missing_ok=True)
np.random.seed(0)                                # same noise at every run

In [ ]:
theta      = [0.25, 0.25, 0.88, 0.09]     # TRUE values: mu_max [1/h], Ks [g/L], Yxs [g/g], m [1/h]
theta_n    = [0.30, 0.30, 1.056, 0.108]   # starting guess (1.2 x true)
theta_mins = [0.125, 0.125, 0.44, 0.045]  # bounds
theta_maxs = [0.375, 0.375, 1.32, 0.135]
NAMES = ['mu_max', 'Ks', 'Yxs', 'm']

system = {
    't_s': [0, 40],          # 40-h fed batch
    't_r': 0.4 if QUICK else 0.16,   # time grid [h] (coarser in a quick check)
    't_d': 0,
    'tvi': {                 # inputs that change during the run
        'u1': {'min': 0.05, 'max': 0.2, 'cvp': 'CPF', 'stps': 5, 'const': 'rel', 'offl': 0.01, 'offt': 0.5},  # dilution rate [1/h], steps
        'u2': {'min': 5, 'max': 35, 'cvp': 'LPF', 'stps': 5, 'const': 'rel', 'offl': 1, 'offt': 0.5},        # feed substrate [g/L], ramps
    },
    'tii': {'y10': {'min': 1, 'max': 5.5}},                    # initial biomass [g/L]
    'tvo': {                 # measured outputs: 10 samples each, at least 0.5 h apart, noise sd 0.2 g/L
        'y1': {'meas': True, 'unc': 0.2, 'init': 0, 'sp': 10, 'samp_s': 1, 'samp_f': [], 'offt': 0.5},     # biomass
        'y2': {'meas': True, 'unc': 0.2, 'init': 0.01, 'sp': 10, 'samp_s': 1, 'samp_f': [], 'offt': 0.5},  # substrate
    },
    'tio': {},
}

models = {
    'can_m': ['MI'], 'krt': {'MI': 'pys'}, 'src': {'MI': str(HERE / 'model.py')}, 'creds': {'MI': ''},
    'theta': {'MI': list(theta_n)}, 't_l': {'MI': theta_mins}, 't_u': {'MI': theta_maxs},
    'mutation': {'MI': [True] * 4}, 'theta_names': {'MI': NAMES},
}
insilicos = {'tr_m': 'MI', 'theta': theta, 'errt': 'abs', 'plt': True,
             'prels': {'1': {'u1': 0.05, 'u2': 30, 'y10': 5.5},
                       '2': {'u1': 0.15, 'u2': 20, 'y10': 2.0}}}

iden_opt = {
    'meth': 'SLSQP', 'ms': False,           # local gradient-based optimiser, one start
    'ob': 'WLS',                            # weighted least squares
    'maxit': 60 if QUICK else 500, 'tol': 1e-8,
    'var-cov': 'M',                         # covariance from the Fisher information matrix
    'sens_m': 'central', 'eps': 1e-3,       # sensitivities by central differences, fixed step
    'c_plt': True, 'f_plt': True, 'plt_s': True, 'log': False,
}

def show_round(unc, model='MI'):
    """Estimates, 95 % CI and t-values of one model, next to the true values."""
    r = unc['results'][model]
    act = np.asarray(r['optimization_result']['activeparams'], bool)
    ci = np.full(len(r['estimations']), np.nan); ci[act] = r['CI']
    tv = np.full(len(r['estimations']), np.nan); tv[act] = r['t_values']
    return pd.DataFrame({'estimate': r['estimations'], '± 95 % CI': ci, 't-value': tv}).round(4)

### Round 1: two preliminary runs at constant inputs

In [ ]:
round_data = {}
expera(system, models, insilicos, {}, expr=1)
expera(system, models, insilicos, {}, expr=2)
resultpr = parmest(system, models, iden_opt)
unc = uncert(resultpr, system, models, iden_opt)
save_rounds(1, unc['results'], 'preliminary', round_data, models, iden_opt, unc['obs'], system)
before = show_round(unc)
before

### Four designs, one per criterion

`des_opt['priors'] = True` adds the information already collected, so the design completes the
existing data instead of repeating it. `PS` is pattern search; `num_parallel_runs` starts it from
several points in parallel and keeps the best.

In [ ]:
def des_settings(criterion):
    return {'pp_ob': criterion,
            'meth': 'PS',                 # 'PS' pattern search, 'DE' differential evolution, 'DEPS' both
            'eps': 1e-3,                  # finite-difference step of the sensitivities
            'priors': True,               # add the information of the data so far
            'plt': True,                  # draw the designed run
            'itr': {'maxpp': 15 if QUICK else 100, 'tolpp': 1, 'pps': 30}}

designs = {}
for crit in (['D'] if QUICK else ['D', 'A', 'E', 'ME']):
    print(f'---------------- {crit}-optimal')
    designs[crit] = mbdoe_pp(des_settings(crit), system, models, round=2,
                             num_parallel_runs=1 if QUICK else 4)
pd.DataFrame({c: {'initial biomass y10': d['tii'].get('y10'), 'sampling times': np.round(d['St'][next(iter(d['St']))], 1)}
              for c, d in designs.items()}).T

The criteria agree on the broad shape (start with much biomass, change the feed where the
growth rate is most sensitive) and differ in details: E and ME push towards the poorest direction
(often the maintenance `m`), D spreads the effort.

### Round 2: run the D-optimal design and refit

In [ ]:
d = designs['D']
expera(system, models, insilicos, d, expr=3, swps=d['swps'])
resultpr = parmest(system, models, iden_opt)
unc = uncert(resultpr, system, models, iden_opt)
save_rounds(2, unc['results'], 'MBDoE-PP (D)', round_data, models, iden_opt, unc['obs'], system)
after = show_round(unc)
ci = pd.concat({'after 2 preliminary runs': before['± 95 % CI'], '+ 1 designed run': after['± 95 % CI']}, axis=1)
ci.set_axis(NAMES).assign(true=theta)

**Try:** run the E-optimal design instead and compare which interval shrinks most; set
`'priors': False` to design as if nothing were known.